# Report H1 2025's conversion rate

*Data Analysis in Python · Topic 4.4*

Three people have sent three different conversion rates for January to June 2025. Work out where each one comes from, and which one belongs in the report.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `mini` table (three days typed in) and the cell that builds the `sessions` table: the clean sessions for January to June 2025, with lesson 3.'s steps.
2. In the first cell marked ✏️, add a `conversion_rate` column to the `mini` table, and create:
   - `mini_mean_rate`: the mean of the `mini` table's three conversion rates
   - `mini_pooled_rate`: the three days' transactions over their sessions
3. In the second cell marked ✏️, create the `daily` table: one row per date in the `sessions` table, with the columns `sessions` and `transactions`, the day's totals, and `conversion_rate`, the day's transactions over its sessions. Then create:
   - `pooled_rate`: the six months' transactions over their sessions
   - `mean_daily_rate`: the mean of the `daily` table's `conversion_rate` column
   - `weighted_rate`: the mean of the `daily` table's `conversion_rate` column, weighted by its `sessions` column
4. In the last cell marked ✏️, create:
   - `mean_row_rate`: the mean of each row's transactions over its sessions, in the `sessions` table
   - `skipped_rows`: the number of rows of the `sessions` table whose rate is `NaN`, which `mean()` left out
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `mini_mean_rate` is about `0.0467` and `mini_pooled_rate` is `0.0256`. The `daily` table has 181 rows. `pooled_rate` is about `0.02538`, `mean_daily_rate` about `0.02496` and `weighted_rate` the same as `pooled_rate`. `mean_row_rate` is about `0.01853`, and `skipped_rows` is `1185`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06"]
START, END = "2025-01-01", "2025-06-30"

In [ ]:
mini = pd.DataFrame({
    "date": pd.to_datetime(["2025-01-06", "2025-01-07", "2025-01-08"]),
    "sessions": [1000, 200, 50],
    "transactions": [20, 8, 4],
})
mini

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

## Three days

In [ ]:
mini["conversion_rate"] = mini["transactions"] / mini["sessions"]
mini_mean_rate = mini["conversion_rate"].mean()
mini_pooled_rate = mini["transactions"].sum() / mini["sessions"].sum()
mini_mean_rate, mini_pooled_rate

## Six months

In [ ]:
daily = sessions.groupby("date")[["sessions", "transactions"]].sum()
daily["conversion_rate"] = daily["transactions"] / daily["sessions"]
pooled_rate = sessions["transactions"].sum() / sessions["sessions"].sum()
mean_daily_rate = daily["conversion_rate"].mean()
weighted_rate = (daily["conversion_rate"] * daily["sessions"]).sum() / daily["sessions"].sum()
pooled_rate, mean_daily_rate, weighted_rate

In [ ]:
row_rates = sessions["transactions"] / sessions["sessions"]
mean_row_rate = row_rates.mean()
skipped_rows = row_rates.isna().sum()
mean_row_rate, skipped_rows

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(name):
        """Return the variable if it's a number, or None."""
        value = found.get(name)
        return value if pd.api.types.is_number(value) and not isinstance(value, bool) else None

    tables = [name for name in ["mini_mean_rate", "mini_pooled_rate", "pooled_rate", "mean_daily_rate", "weighted_rate", "mean_row_rate", "skipped_rows"]
              if isinstance(found.get(name), (pd.DataFrame, pd.Series))]
    if tables:
        results.append((False, f"{', '.join(tables)} should be numbers, not Series or tables. Each one is a single value, such as mini[\"conversion_rate\"].mean()."))

    mini_mean, mini_pooled = number("mini_mean_rate"), number("mini_pooled_rate")
    if mini_mean is None or mini_pooled is None:
        results.append((False, "mini_mean_rate and mini_pooled_rate should be numbers. Add mini's conversion_rate column first."))
    elif abs(mini_mean - 0.0256) < 1e-9 and abs(mini_pooled - 0.046667) < 0.000001:
        results.append((False, "mini_mean_rate and mini_pooled_rate are swapped: the mean of the three rates is 0.0467, and 32 transactions in 1,250 sessions is 0.0256."))
    elif abs(mini_mean - 0.046667) > 0.000001:
        results.append((False, f"mini_mean_rate should be the mean of 2%, 4% and 8%, about 0.0467. It is {mini_mean!r}."))
    elif abs(mini_pooled - 0.0256) > 1e-9:
        results.append((False, f"mini_pooled_rate should be 32 transactions over 1,250 sessions: 0.0256. It is {mini_pooled!r}."))
    else:
        results.append((True, "Three days: the mean of the rates is 4.67%, but 32 transactions in 1,250 sessions is 2.56%. The 50-session day counted as much as the 1,000-session day."))

    sessions = found.get("sessions")
    daily = found.get("daily")
    if not isinstance(sessions, pd.DataFrame) or len(sessions) != 10612 or sessions["sessions"].sum() != 611934:
        results.append((False, "sessions should be the clean table for January to June 2025. Run the cell that builds it, and don't filter it."))
    elif not isinstance(daily, pd.DataFrame):
        results.append((False, "daily isn't a DataFrame yet. Group sessions by date, total sessions and transactions, and add conversion_rate."))
    elif len(daily) != 181:
        results.append((False, f"daily has {len(daily):,} rows, but January to June 2025 has 181 days. Group by date only."))
    elif "conversion_rate" not in daily.columns or not ((daily["conversion_rate"] - daily["transactions"] / daily["sessions"]).abs() < 1e-12).all():
        results.append((False, "daily's conversion_rate should be each day's transactions over its sessions, after totaling the day."))
    else:
        results.append((True, "daily has the 181 days, each with its own conversion rate."))

    pooled, mean_daily, weighted = number("pooled_rate"), number("mean_daily_rate"), number("weighted_rate")
    if pooled is None or mean_daily is None or weighted is None:
        results.append((False, f"pooled_rate, mean_daily_rate and weighted_rate should be numbers. They are {found.get('pooled_rate')!r}, {found.get('mean_daily_rate')!r} and {found.get('weighted_rate')!r}."))
    elif abs(pooled - 0.024956) < 0.000001:
        results.append((False, "pooled_rate is 2.50%, the mean of the daily rates. The pooled rate is the six months' total transactions over their total sessions."))
    elif abs(pooled - 0.025379) > 0.000001:
        results.append((False, f"pooled_rate should be sessions[\"transactions\"].sum() / sessions[\"sessions\"].sum(), about 0.02538. It is {pooled!r}."))
    elif abs(mean_daily - 0.024956) > 0.000001:
        results.append((False, f"mean_daily_rate should be daily[\"conversion_rate\"].mean(), about 0.02496. It is {mean_daily!r}."))
    elif abs(weighted - 0.024956) < 0.000001:
        results.append((False, "weighted_rate is the plain mean of the daily rates. Weight each rate by its day's sessions: (daily[\"conversion_rate\"] * daily[\"sessions\"]).sum() / daily[\"sessions\"].sum()."))
    elif abs(weighted - pooled) > 1e-12:
        results.append((False, f"weighted_rate should equal pooled_rate when each day's rate is weighted by its sessions. It is {weighted!r}."))
    else:
        results.append((True, f"The six months converted {pooled:.2%}. The mean of the daily rates says {mean_daily:.2%}, and weighting the days by their sessions gives the pooled rate back."))

    mean_row, skipped = number("mean_row_rate"), number("skipped_rows")
    if mean_row is None or skipped is None:
        results.append((False, f"mean_row_rate and skipped_rows should be numbers. They are {found.get('mean_row_rate')!r} and {found.get('skipped_rows')!r}."))
    elif abs(mean_row - 0.018527) > 0.000001:
        results.append((False, f"mean_row_rate should be (sessions[\"transactions\"] / sessions[\"sessions\"]).mean(), about 0.01853. It is {mean_row!r}."))
    elif skipped == 0:
        results.append((False, "skipped_rows is 0. Count the NaN rates: (sessions[\"transactions\"] / sessions[\"sessions\"]).isna().sum(). The refund rows have 0 sessions and 0 transactions."))
    elif skipped != 1185:
        results.append((False, f"skipped_rows should be the number of NaN rates, 1,185. It is {skipped!r}."))
    else:
        results.append((True, f"The mean of the rows says {mean_row:.2%}, and it skipped 1,185 refund rows whose 0 / 0 is NaN. Report the pooled rate."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
mini["conversion_rate"] = mini["transactions"] / mini["sessions"]
mini_mean_rate = mini["conversion_rate"].mean()
mini_pooled_rate = mini["transactions"].sum() / mini["sessions"].sum()
mini_mean_rate, mini_pooled_rate
```

```python
daily = sessions.groupby("date")[["sessions", "transactions"]].sum()
daily["conversion_rate"] = daily["transactions"] / daily["sessions"]
pooled_rate = sessions["transactions"].sum() / sessions["sessions"].sum()
mean_daily_rate = daily["conversion_rate"].mean()
weighted_rate = (daily["conversion_rate"] * daily["sessions"]).sum() / daily["sessions"].sum()
pooled_rate, mean_daily_rate, weighted_rate
```

```python
row_rates = sessions["transactions"] / sessions["sessions"]
mean_row_rate = row_rates.mean()
skipped_rows = row_rates.isna().sum()
mean_row_rate, skipped_rows
```

</details>